# Openpose 1.7.0 Demo

In [1]:
# 最初に、ランタイムの種類を「GPU」に変更してください。（ランタイム＞ランタイムのタイプを変更）
# GPUが有効になっていることを確認します。

# First, change the runtime type to "GPU".
# Confirm runtime to GPU
! nvcc --version
! nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Tue_Aug_15_22:02:13_PDT_2023
Cuda compilation tools, release 12.2, V12.2.140
Build cuda_12.2.r12.2/compiler.33191640_0
Wed Oct 23 04:08:57 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   65C    P8              11W /  7

In [2]:
# Openpose バージョン指定タグ
# Openpose version tag
ver_openpose = "v1.7.0"
! echo $ver_openpose

v1.7.0


In [3]:
# CMakeが古いとOpenpose（CUDA10)が失敗するので、バージョンを確認します。（BugfixはCMake 3.12.3）
# If CMake is old, Openpose build fails, so download the latest version
# https://developercommunity.visualstudio.com/content/problem/354325/cmake-project-cannot-detect-cuda-10.html
! cmake --version

cmake version 3.30.4

CMake suite maintained and supported by Kitware (kitware.com/cmake).


In [5]:
# 2021.03.03時点の最新CMakeを再ビルド（15分くらい）
# Rebuild the latest CMake as of 2021.03.03 (about 15 minutes)
! wget -c "https://github.com/Kitware/CMake/releases/download/v3.19.6/cmake-3.19.6.tar.gz"
! tar xf cmake-3.19.6.tar.gz
! cd cmake-3.19.6 && ./configure && make && sudo make install

Streaming output truncated to the last 5000 lines.
g++        -DCMAKE_BOOTSTRAP    -DCMake_HAVE_CXX_MAKE_UNIQUE=1 -DCMake_HAVE_CXX_FILESYSTEM=1   -I/content/cmake-3.19.6/Bootstrap.cmk   -I/content/cmake-3.19.6/Source   -I/content/cmake-3.19.6/Source/LexerParser   -I/content/cmake-3.19.6/Utilities/std   -I/content/cmake-3.19.6/Utilities  -c /content/cmake-3.19.6/Source/cmTryRunCommand.cxx -o cmTryRunCommand.o
g++        -DCMAKE_BOOTSTRAP    -DCMake_HAVE_CXX_MAKE_UNIQUE=1 -DCMake_HAVE_CXX_FILESYSTEM=1   -I/content/cmake-3.19.6/Bootstrap.cmk   -I/content/cmake-3.19.6/Source   -I/content/cmake-3.19.6/Source/LexerParser   -I/content/cmake-3.19.6/Utilities/std   -I/content/cmake-3.19.6/Utilities  -c /content/cmake-3.19.6/Source/cmUnsetCommand.cxx -o cmUnsetCommand.o
g++        -DCMAKE_BOOTSTRAP    -DCMake_HAVE_CXX_MAKE_UNIQUE=1 -DCMake_HAVE_CXX_FILESYSTEM=1   -I/content/cmake-3.19.6/Bootstrap.cmk   -I/content/cmake-3.19.6/Source   -I/content/cmake-3.19.6/Source/LexerParser   -I/content/cmake

In [4]:
# ライブラリインストール
# Install library

# Basic
! sudo apt-get --assume-yes update
! sudo apt-get --assume-yes install build-essential
# OpenCV
! sudo apt-get --assume-yes install libopencv-dev
# General dependencies
! sudo apt-get --assume-yes install libatlas-base-dev libprotobuf-dev libleveldb-dev libsnappy-dev libhdf5-serial-dev protobuf-compiler
! sudo apt-get --assume-yes install --no-install-recommends libboost-all-dev
# Remaining dependencies, 14.04
! sudo apt-get --assume-yes install libgflags-dev libgoogle-glog-dev liblmdb-dev
# Python3 libs
! sudo apt-get --assume-yes install python3-setuptools python3-dev build-essential
! sudo apt-get --assume-yes install python3-pip
! sudo -H pip3 install --upgrade numpy protobuf opencv-python
# OpenCL Generic
! sudo apt-get --assume-yes install opencl-headers ocl-icd-opencl-dev
! sudo apt-get --assume-yes install libviennacl-dev

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Ign:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,068 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,162 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,412 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu

In [6]:
# Openposeのコードをclone
# Clone Openpose
! git clone  --depth 1 -b "$ver_openpose" https://github.com/CMU-Perceptual-Computing-Lab/openpose.git

Cloning into 'openpose'...
remote: Enumerating objects: 659, done.
remote: Counting objects: 100% (659/659), done.
remote: Compressing objects: 100% (625/625), done.
remote: Total 659 (delta 171), reused 162 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (659/659), 46.51 MiB | 25.51 MiB/s, done.
Resolving deltas: 100% (171/171), done.
Note: switching to '8ca5c1d95a42340b323e9273654d1db98bec779c'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



In [7]:
# build用ディレクトリを作成
# Create build directory
! cd openpose && mkdir build && cd build

In [8]:
# https://github.com/CMU-Perceptual-Computing-Lab/openpose/blob/master/doc/installation.md#cmake-command-line-configuration-ubuntu-only
# 上記インストール手順のシナリオ１でインストール実行
# Scenario 1 - Caffe not installed and OpenCV installed using apt-get
! cd openpose/build && cmake ..

# COCOモデルのDLオプション付き(「# ! cd」の部分を「! cd」に変更してください。)
# If you want to download the COCO model as well, execute the following command. (Please change "# ! cd" part to "! cd".)
# ! cd openpose/build && cmake .. -D DOWNLOAD_BODY_COCO_MODEL=ON

# MPIモデルのDLオプション付き（同上）
# If you want to download the MPI model as well, execute the following command.
# ! cd openpose/build && cmake .. -D DOWNLOAD_BODY_MPI_MODEL=ON

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- GCC detected, adding compile flags
-- GCC detected, adding compile flags
-- Looking for pthread.h
-- Looking for pthread.h - found
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE  
-- Found CUDA: /usr/local/cuda (found version "12.2") 
-- Building with CUDA.
-- CUDA detected: 12.2
-- Found cuDNN: ver. 8.9.6 found (include: /usr/include, library: /usr/lib/x86_64-linux-gnu/libcudnn.so)
-- Added CUDA NVCC flags for: sm_7

In [9]:
# Openposeのビルド（15分くらい）
# Openpose Building
! cd openpose/build && make -j`nproc`
# outputフォルダ作成
! cd openpose && mkdir output

make: *** No targets specified and no makefile found.  Stop.


In [10]:
# デモ動画を解析
# 出力された解析結果は、Colab画面の左メニューのフォルダアイコンから「openpose/output」以下に配置されます。
# Run and check the sample
# The output analysis result is placed under "openpose/output" from the folder icon on the left menu of the Colab screen.
! cd openpose && ./build/examples/openpose/openpose.bin --video examples/media/video.avi --display 0  --write_video ./output/openpose.avi

/bin/bash: line 1: ./build/examples/openpose/openpose.bin: No such file or directory


In [11]:
# デモ動画の結果をffmpegで再エンコードして表示します。少し時間がかかります。
# The result of the demo video is re-encoded with ffmpeg and displayed. It will take some time.
! ffmpeg -i "./openpose/output/openpose.avi" -c:v libvpx-vp9 -b:v 1000K -r 30 -threads 8 -speed 1 -tile-columns 6 -frame-parallel 1 -auto-alt-ref 1 -lag-in-frames 25 -c:a libopus -b:a 64k -f webm -y out.webm

from IPython.display import HTML
import base64
import io

video = io.open('./out.webm', 'r+b').read()
encoded = base64.b64encode(video)
data  = '''
<video id="out" width="1280" height="720" controls><source src="data:video/mp4;base64,{0}" type="video/mp4" /></video>
'''.format(encoded.decode('ascii'))

HTML(data)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

FileNotFoundError: [Errno 2] No such file or directory: './out.webm'

In [9]:
from google.colab import drive
import os

# Googleドライブマウント
# マウントすると、アクセス許可が求められるので、許可して、キーをコピーして、Colab側の入力欄にCtrl+Vで貼り付けて、Enter。
# mount Google Drive
# When you mount it, you will be asked for permission, so allow it, copy the key, paste(Ctrl+V) it in the input field on the Colab side, and press Enter.
drive.mount('/gdrive')

# Google Drive のルートディレクトリ
# Google Drive root directory
root_path = "/gdrive/My Drive/"

# Googleドライブ上で解析したい動画パス（この値を好きな値に変更する事で、ご自分のGoogleドライブ上の動画を解析できます）
# Video path that you want to analyze on Google Drive (You can analyze the video on your own Google Drive by changing this value to a value you like)
file_path = "a.mp4"

# Googleドライブ上の動画を解析（Googleドライブのルートディレクトリに解析結果の「openpose.avi」が出力されます）
# Analyze video on Google Drive (The analysis result "openpose.avi" is output to the root directory of Google Drive)
! cd openpose && ./build/examples/openpose/openpose.bin --video "$root_path/$file_path" --display 0  --write_video "$root_path/openpose.avi"

Mounted at /gdrive
/bin/bash: line 1: cd: openpose: No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
# Googleドライブ上の解析結果動画をffmpegで再エンコードして表示します。少し時間がかかります。
# The analysis result video on Google Drive is re-encoded with ffmpeg and displayed. It will take some time.
! ffmpeg -i "a.mp4" -c:v libvpx-vp9 -b:v 1000K -r 30 -threads 8 -speed 1 -tile-columns 6 -frame-parallel 1 -auto-alt-ref 1 -lag-in-frames 25 -c:a libopus -b:a 64k -f webm -y out.webm

from IPython.display import HTML
import base64
import io

video = io.open('./out.webm', 'r+b').read()
encoded = base64.b64encode(video)
data  = '''
<video id="out" width="1280" height="720" controls><source src="data:video/mp4;base64,{0}" type="video/mp4" /></video>
'''.format(encoded.decode('ascii'))

HTML(data)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [8]:
# 指定した人数分のみ解析したい場合、「number_people_max」の後ろに人数を指定して下さい
# If you want to analyze only the specified number of people, please specify the number of people after "number_people_max"
# 結果はひとつ上のコードを再実行する事でブラウザ上で確認できます。
# The result can be confirmed on the browser by re-executing the code one level above.
! cd openpose && ./build/examples/openpose/openpose.bin --video "$root_path/$file_path" --display 0  --write_video "/content/a.mp4" --number_people_max 1

/bin/bash: line 1: cd: openpose: No such file or directory


In [4]:
# 手の解析も一緒に行いたい場合、「--hand」を追加してください
# If you also want to analyze the hands together, add "--hand"
# 結果は２つ上のコードを再実行する事でブラウザ上で確認できます。
# The result can be confirmed on the browser by re-executing the code two level above.
! cd openpose && ./build/examples/openpose/openpose.bin --video "a.mp4" --display 0  --write_video "$root_path/openpose.avi" --hand

/bin/bash: line 1: cd: openpose: No such file or directory


In [ ]:
# 顔の解析も一緒に行いたい場合、「--face」を追加してください
# If you also want to analyze the hands together, add "--hand"
# 結果は３つ上のコードを再実行する事でブラウザ上で確認できます。
# The result can be confirmed on the browser by re-executing the code three level above.
! cd openpose && ./build/examples/openpose/openpose.bin --video "a.mp4" --display 0  --write_video "$root_path/openpose.avi" --face